In [2]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


### Initialize the LLM

In [ ]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0,
    max_tokens=12000
)

### Create a simple agent

#### Create tool

In [3]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Create agent

In [4]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
agent1 = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [5]:

response =agent1.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": 
                """
                What is the weather in Shanghai?
                """
            }
        ]
    }
)
print(response)


{'messages': [HumanMessage(content='\n                What is the weather in Shanghai?\n                ', additional_kwargs={}, response_metadata={}, id='02337d4f-f7a1-42e6-9d20-732a8b30711d'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yHpUXGyYyGRNwdpEaRrWJZ7F', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 82, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-Cq8D6wZcvdaqAYBmV4S5XERpU0fKJ', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b4e07-2d49-7c41-8c07-15b6d2541c22-0', tool_calls=[{'name': 'get_weather', 'a

### Create function to display agent message

In [89]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  


In [90]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        #"assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
        print("\nAassistant reply:")
        print(assistant_text or "")
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




In [9]:
messages = invoke_agent_messages(
    agent1,
    "What is the weather in Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "What is the weather in Shanghai?",
  "tool_name": "get_weather",
  "tool_output": "Weather in Shanghai: Sunny, 72°F"
}

Aassistant reply:
The weather in Shanghai is currently sunny with a temperature of 72°F.


### New tool - get online image content from its url

#### Create function

In [10]:
def get_imagedetail(image_url):
    
    # Create messages including both text and image input
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe the image in detail."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url  # Use the direct image URL
                    }
                }
            ]
        }
    ]
    
    response = model.invoke(messages)
    return response.content


Optional: test the function

In [11]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"
response=get_imagedetail(image_url)

print(response)

The image shows a sleek, modern BMW car driving on a road. The car is silver with a prominent front grille featuring the BMW logo. The headlights are on, emitting a bright light. The license plate reads "M IA 1272E." The car is captured from a low angle, emphasizing its dynamic and sporty design. In the background, there is a blurred view of rocky terrain and a clear blue sky, suggesting motion and speed. The overall composition highlights the car's elegance and performance capabilities.


#### Create a tool

In [12]:
from langchain.tools import tool 

@tool
def get_img(url: str) -> str:
    """Get the description of the image in the url given by user."""
    return get_imagedetail(url)

#### Create a new agent

In [13]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
agent2 = create_agent(
    model,
    tools = [search, get_weather,get_img],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Test and compare agents

In [14]:
response = model.invoke(
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print(response.content)

I'm unable to view or describe images directly from URLs. However, if you can provide a description or key elements from the image, I'd be happy to help analyze or discuss it!


In [15]:
messages = invoke_agent_messages(
    agent1, 
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print_message_pairs(messages,verbose=True)

{
  "user_query": "\n    Describe the image in the below url:\n    \"https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4\"\n    ",
  "tool_name": "search",
  "tool_output": "Results for: https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
}

Aassistant reply:
I cannot directly view or describe images from URLs. You might want to check the URL in a web browser to see the image. If you have any other questions or need further assistance, feel free to ask!


In [16]:
messages = invoke_agent_messages(
    agent2, 
    """
    Describe the image in the below url:
    "https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4"
    """
)
print_message_pairs(messages,verbose=True)

{
  "user_query": "\n    Describe the image in the below url:\n    \"https://s.yimg.com/os/creatr-uploaded-images/2022-10/64609b80-4969-11ed-afec-2f62147a62e4\"\n    ",
  "tool_name": "get_img",
  "tool_output": "The image features a sleek, blue BMW car parked in a scenic outdoor setting. The car has a modern design with a prominent front grille and sharp, angular headlights. The BMW logo is visible on the hood. The car's body has smooth curves and aerodynamic lines, contributing to its sporty appearance. The wheels are large with intricate alloy rims. In the background, there is a picturesque landscape with rolling hills, a body of water, and some trees, all under a clear blue sky. The setting suggests a peaceful, open area, possibly a countryside or lakeside location."
}

Aassistant reply:
The image features a sleek, blue BMW car parked in a scenic outdoor setting. The car has a modern design with a prominent front grille and sharp, angular headlights, and the BMW logo is visible on 

### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [136]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [138]:
query="Search for the explaination of context_schemaand and mmiddleware in Langchain Agent, and then interpret them further."

In [139]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

I wasn't able to retrieve specific information about "context_schema" and "middleware" in Langchain Agent. However, I can provide a general explanation based on typical usage in programming and AI frameworks:

1. **Context Schema**:
   - In many frameworks, a context schema refers to the structure or format of the data that is passed around within the system. It defines what kind of information is included, how it is organized, and how it can be accessed or modified. In the context of Langchain or similar AI frameworks, a context schema might specify the types of inputs and outputs that an agent can handle, including any metadata or additional parameters that are necessary for processing.

2. **Middleware**:
   - Middleware generally refers to software that acts as a bridge between different systems or layers within an application. It can be used to manage data flow, handle requests, or perform operations like logging, authentication, and error handling. In the context of Langchain Age

In [140]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

It seems there was an issue retrieving the specific explanations for "context_schema" and "middleware" in Langchain Agent. However, I can provide a general interpretation based on typical usage in similar contexts:

### Context Schema in Langchain Agent

**Context Schema** typically refers to a structured format or blueprint that defines the context in which an agent operates. In the context of Langchain or similar frameworks, a context schema might include:

- **Variables and Parameters**: Definitions of the variables that the agent can use or modify during its operation.
- **Data Types**: Specifications of the types of data (e.g., strings, integers, objects) that the agent can handle.
- **Constraints**: Rules or conditions that the data must satisfy.
- **Relationships**: How different pieces of data relate to each other within the context.

In Langchain, a context schema would help in structuring the input and output data for agents, ensuring that they operate within defined paramete

### MiddleWare

#### Setup: model + tools

In [3]:
from langchain_core.tools import tool
# --- Define tools ---
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"



tools = [search, get_weather]

# --- Define model (replace your API key/config as needed) ---
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=4800
)

#### Define custom Context + middleware

In [4]:

from typing import TypedDict, Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

class Context(TypedDict):
    user_preferences: dict  # {"style": "...", "verbosity": "..."}

class CustomMiddleware(AgentMiddleware):
    # (Optional) stage-specific tool restrictions:
    # tools = [tool1, tool2]

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        # Read preferences from runtime context
        prefs = runtime.context.get("user_preferences", {}) or {}
        style = str(prefs.get("style", "general")).lower()
        verbosity = str(prefs.get("verbosity", "normal")).lower()

        # Base prompt
        system_prompt = "You are a helpful assistant."

        # Style-specific guidance
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal, approachable, and friendly."

        # Verbosity-specific guidance
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points; use short sentences and bullet points where helpful."

        # Tune generation params (optional)
        temperature = 0.2 if style == "technical" else 0.7  # more deterministic for technical, more open for casual

        # Return updates for the upcoming model call
        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": temperature},
        }


#### Create agent

In [5]:

agent = create_agent(
    model,
    tools=tools,                       # e.g., [search, get_weather]
    middleware=[CustomMiddleware()],
    context_schema=Context,            # <-- use context, not state
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)


#### Invoke the agent with user_preferences

In [6]:
query="Search for the explaination vector embeddings." 
query="Search for the story lines and theme in 三国演义 in Chinese" 
query="Search for the story lines and theme in Games of Throne and then introduce these in Chinese." 

In [9]:
# A user who prefers technical & detailed responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "technical",
            "verbosity": "detailed"
        }
    }
)
messages = result["messages"]

print("\n=============================== Assistant reply (technical + detailed) =================================")
print_message_pairs(messages,verbose=True)



=============================== Assistant reply (technical + detailed) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

**Storylines:**
1. **The Iron Throne:** The central storyline revolves around the struggle for power and control of the Iron Throne of the Seven Kingdoms. Various noble families, including the Starks, Lannisters, Baratheons, and Targaryens, vie for dominance.
   
2. **The Stark Family:** The Starks of Winterfell face numerous challenges, including betrayal, political intrigue, and the fight to reclaim their home and honor.

3. **Daenerys Targaryen's Quest:** Daenerys Targaryen's journey from exile to power, as she seeks to reclaim the throne for her family, is marked by her growth as a leader and her acquisition 

In [130]:

# A user who prefers casual & brief responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "casual",
            "verbosity": "brief"
        }
    }
)
messages = result["messages"]


print("\n=============================== Assistant reply (casual + brief) =================================")
print_message_pairs(messages,verbose=True)




=============================== Assistant reply (casual + brief) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

- **Storylines:**
  - **Power Struggles:** The series revolves around the battle for the Iron Throne among noble families.
  - **Family Dynamics:** Focuses on the relationships and conflicts within families like the Starks, Lannisters, and Targaryens.
  - **Mystical Elements:** Includes dragons, magic, and the threat of the White Walkers.
  - **Political Intrigue:** Features alliances, betrayals, and complex political maneuvers.

- **Themes:**
  - **Power and Ambition:** Explores the lengths people go to gain and maintain power.
  - **Loyalty and Betrayal:** Highlights the importance and consequences of loyalty and bet

## Multi-agent: 1. Subagents  

### Build a personal assistant with subagents

https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant

In [23]:
"""
Personal Assistant Supervisor Example

This example demonstrates the tool calling pattern for multi-agent systems.
A supervisor agent coordinates specialized sub-agents (calendar and email)
that are wrapped as tools.
"""

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# ============================================================================
# Step 1: Define low-level API tools (stubbed)
# ============================================================================

@tool
def create_calendar_event(
    title: str,
    start_time: str,  # ISO format: "2024-01-15T14:00:00"
    end_time: str,    # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],      # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    return ["09:00", "14:00", "16:00"]


# ============================================================================
# Step 2: Create specialized sub-agents
# ============================================================================

from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=(
        "You are a calendar scheduling assistant. "
        "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
        "into proper ISO datetime formats. "
        "Use get_available_time_slots to check availability when needed. "
        "Use create_calendar_event to schedule events. "
        "Always confirm what was scheduled in your final response."
    )
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=(
        "You are an email assistant. "
        "Compose professional emails based on natural language requests. "
        "Extract recipient information and craft appropriate subject lines and body text. "
        "Use send_email to send the message. "
        "Always confirm what was sent in your final response."
    )
)

# ============================================================================
# Step 3: Wrap sub-agents as tools for the supervisor
# ============================================================================

@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


# ============================================================================
# Step 4: Create the supervisor agent
# ============================================================================

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=(
        "You are a helpful personal assistant. "
        "You can schedule calendar events and send emails. "
        "Break down user requests into appropriate tool calls and coordinate the results. "
        "When a request involves multiple actions, use multiple tools in sequence."
    )
)

# ============================================================================
# Step 5: Use the supervisor
# ============================================================================

if __name__ == "__main__":
    # Example: User request requiring both calendar and email coordination
    user_request = (
        "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
        "and send them an email reminder about reviewing the new mockups."
    )

    print("User Request:", user_request)
    print("\n" + "="*80 + "\n")

    for step in supervisor_agent.stream(
        {"messages": [{"role": "user", "content": user_request}]}
    ):
        for update in step.values():
            for message in update.get("messages", []):
                message.pretty_print()

User Request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.


================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_4lDsMPSYlNlQwGoIrjnR4AsG)
 Call ID: call_4lDsMPSYlNlQwGoIrjnR4AsG
  Args:
    request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_ssoTL3JIW2jnCuIUE5a17KK7)
 Call ID: call_ssoTL3JIW2jnCuIUE5a17KK7
  Args:
    request: Send the design team an email reminder about reviewing the new mockups
================================= Tool Message =================================
Name: manage_email

I have sent an email to the design team reminding them to review the new mockups.
================================= Tool Message =================================
Name: schedule_event

The meeting with the design team has been scheduled for next Tuesday, November 7th, from 2:00 PM to 3:00 PM.

### Build an own agent

#### New functions for tools

##### Function: Get text from website

In [ ]:

from openai import OpenAI
import requests
from bs4 import BeautifulSoup

def extract_article_text(url: str) -> str:
    """Fetch and extract main text from a webpage."""
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Try to get main content
    article_tags = soup.find_all(["article", "p"])
    text = " ".join(tag.get_text(strip=True) for tag in article_tags)
    return text[:8000]  # Limit to ~8k chars for token safety


Test the function

In [17]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"

response=extract_article_text(article_url).strip()
print(response)


Die Deutsche Umwelthilfe im Netz Pressemitteilung •	Mehr ultrafeine Feinstaub-Partikel und Stickoxide: DUH-Abgasmessung im realen Straßenbetrieb an Euro-5-Diesel-Pkw widerlegt Mythos von „besonders nachhaltigem“ Dieselkraftstoff HVO100 •	DUH fordert Verkehrsminister Wissing auf, seine Behauptungen zu unterlassen, dass mit HVO100 „lokale Umweltbelastung in Städten und Kommunen“ reduziert werde, der Minister muss ihm vorliegende Untersuchungen über erhöhte Stickoxid-Emissionen zum neuen Dieselkraftstoff veröffentlichen •	DUH fordert Nachrüstung schmutziger Diesel-Pkw und Nutzfahrzeuge auf Kosten der Hersteller statt klima- und gesundheitsschädlicher Pseudo-Alternativen Berlin, 27.6.2024: Der neue Dieselkraftstoff HVO100 soll die klimaschädlichen Treibhausgasemissionen um „bis zu 90 Prozent“ verringern und gleichzeitig die „lokale Umweltbelastung in Städten und Kommunen“ reduzieren – so wirbt Bundesverkehrsminister Wissing für diesen angeblichen Wunderkraftstoff. Messungen der Deutschen U

##### Function: Translate text into target language

In [7]:
def text_translation(target_language: str = "Chinese", text: str="") -> str:
     
    system = (
        f"You are a professional translator. Precisely translate into {target_language}. "
        "Return ONLY the accurate translated text. No explanations, no quotes."
    )
    
    user = f"Target language: {target_language}\n\nText:\n{text}"

    messages=[
        ("system", system),
        ("user",user),
    ]

    # print(messages)
    response = model.invoke(messages)
    return response.content 

Test the function

In [8]:
#The article is written in German
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
sourcetext=extract_article_text(article_url).strip()

#In this function the default target languague is Chinese 
print(text_translation(text=sourcetext))

德国环保援助网络新闻稿
• 更多超细颗粒物和氮氧化物：DUH在实际道路运行中对欧5柴油乘用车的排放测量推翻了“特别可持续”柴油燃料HVO100的神话
• DUH要求交通部长Wissing停止声称使用HVO100可以减少城市和社区的“局部环境污染”，部长必须公布他掌握的关于新柴油燃料氮氧化物排放增加的调查结果
• DUH要求制造商承担费用对污染柴油乘用车和商用车进行改装，而不是使用对气候和健康有害的伪替代品

柏林，2024年6月27日：新柴油燃料HVO100据称可以减少“高达90%”的温室气体排放，同时减少城市和社区的“局部环境污染”——联邦交通部长Wissing如此宣传这一所谓的神奇燃料。然而，德国环保援助（DUH）对一辆欧5柴油乘用车的测量显示：新柴油燃料HVO100比传统柴油更有害健康。DUH自己的排放控制研究所（EKI）的测量显示，与传统柴油相比，使用HVO100时柴油废气毒物NOx的排放增加了20%。ADAC的测量显示，特别有害健康的超细颗粒物数量显著增加。HVO100是一种虚假解决方案，其燃烧和生产过程中常伴有对气候和生物多样性严重的副作用。使用HVO100和其他“替代”燃料不是对根本性交通转型和对现有车辆中污染柴油乘用车进行改装的替代方案。

Axel Friedrich，DUH排放控制研究所所长：“我们对一辆大众途锐欧5的测量显示，使用HVO100时氮氧化物排放比传统柴油高约20%。特别令人担忧的是，超细颗粒物也在增加。这些颗粒物对健康特别有害，因为它们可以深入身体直到血液中。HVO燃料也不合理地被排除在CO2定价之外。这必须立即停止。”HVO100据称仅由旧炸油和其他残留物制成，但实际上显然也由专门种植的植物油如棕榈油制成。原材料的有限供应、掺杂其他有价值原材料的欺诈行为以及棕榈油、大豆等的种植占用的巨大面积导致HVO的使用对气候和生物多样性产生部分严重影响。旧炸油和含油的残留和废弃物供应不足，并且已经作为化学工业中的有价值原材料使用。在使用旧炸油生产HVO100时，工业必须通过原油产品来替代缺失的数量。在诚实的整体评估中，实际的温室气体排放因此往往甚至高于传统柴油燃料。

Jürgen Resch，DUH联邦执行董事：“我们要求联邦交通部长Volker Wissing立即停止关于HVO100柴油在城市和社区中减少环境污染的错误说法。我们还想知道他

In [ ]:
#Test with input target languague
target_language="English"

print(text_translation(target_language="English",text=sourcetext))

The German Environmental Aid on the Net Press Release • More ultrafine particulate matter and nitrogen oxides: DUH emissions measurement in real road operation on Euro-5 diesel cars disproves the myth of "particularly sustainable" diesel fuel HVO100 • DUH calls on Transport Minister Wissing to refrain from claiming that HVO100 reduces "local environmental pollution in cities and municipalities," and the minister must publish studies available to him on increased nitrogen oxide emissions from the new diesel fuel • DUH demands retrofitting of dirty diesel cars and commercial vehicles at the manufacturers' expense instead of climate- and health-damaging pseudo-alternatives Berlin, 27.6.2024: The new diesel fuel HVO100 is supposed to reduce climate-damaging greenhouse gas emissions by "up to 90 percent" while also reducing "local environmental pollution in cities and municipalities" – this is how Federal Transport Minister Wissing promotes this alleged miracle fuel. However, measurements b

##### Function: Get description of a Web imagme

In [19]:
def image_detail(image_url):
    
    # Create messages including both text and image input
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe the image in detail."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url  # Use the direct image URL
                    }
                }
            ]
        }
    ]
    
    response = model.invoke(messages)
    return response.content


Test the function

In [166]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"

response=image_detail(image_url)

print(response)

The image shows a silver BMW car driving on a road with a scenic backdrop. The car is captured from the front, showcasing its sleek design and distinctive kidney grille. The headlights are on, adding to the dynamic appearance. The license plate reads "M IA 1272E." The background features a blurred view of rocky terrain and greenery, suggesting motion and speed. The sky is partly cloudy, adding to the picturesque setting. The overall composition emphasizes the car's sporty and modern aesthetic.


#### Define tools

In [20]:
from langchain.tools import tool

@tool
def get_web_article(
    article_url: str,
) -> str:
    """
    Get the article content from the website url given by user.
    """
    response=extract_article_text(article_url)
    return response


@tool
def get_translation(target_language: str, text: str) -> str:
    """
    Translate given text into `target_language`.
    By default the value of `target_language` is 'Chinese'.
    """
    response=text_translation(target_language,text) 
    target_language="Chinese"
    return response



@tool
def get_web_image(
    image_url: str,
) -> str:
    """
    Get the image content from the website url given by user.
    Describe the image in detail.
    """
    response=image_detail(image_url)
    return response

#### Create specialized sub-agents


##### Creat the web article sub-agent

The agent understands fetch content from the input url and then translate it into target languague(by default Chinese).  

In [21]:
from langchain.agents import create_agent


WEB_ARTICLE_AGENT_PROMPT = (
    "You are a helpful Ai assitant."
    "Help user to get accurate information from the given website url"
    "Use the tool get_web_article to get the article content "
    "Then use the tool get_translation to get the translated version of the content "
    "Do not remove any part"
    "Do not do any summary or ellabaration"
)

web_article_subagent = create_agent(
    model,
    tools=[get_web_article, get_translation],
    system_prompt=WEB_ARTICLE_AGENT_PROMPT,
)

Test the agent to see how it calls the 2 tools

In [29]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"

query = article_url

for step in web_article_subagent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_web_article (call_o2GrtLwyxkPfZcVKitJK6vKq)
 Call ID: call_o2GrtLwyxkPfZcVKitJK6vKq
  Args:
    article_url: https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/
================================= Tool Message =================================
Name: get_web_article

Die Deutsche Umwelthilfe im Netz Pressemitteilung •	Mehr ultrafeine Feinstaub-Partikel und Stickoxide: DUH-Abgasmessung im realen Straßenbetrieb an Euro-5-Diesel-Pkw widerlegt Mythos von „besonders nachhaltigem“ Dieselkraftstoff HVO100 •	DUH fordert Verkehrsminister Wissing auf, seine Behauptungen zu unterlassen, dass mit HVO100 „lokale Umweltbelastung in Städten und Kommunen“ reduziert werde, der Minister muss ihm vorliegende Untersuchungen über erhöhte Stickoxid-Emissionen zum neuen Dieselkraftstoff veröffentlichen 

##### Creat the web image sub-agent

The email agent handles message composition and sending. It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [188]:
WEB_IMAGE_PROMPT = (
    "You are a helpful Ai assitant."
    "Help user to get accurate information from the given website url"
    "Use the tool get_web_image to get the image information"
    "Describe the image in detail." 
)

web_image_subagent = create_agent(
    model,
    tools=[get_web_image],
    system_prompt=WEB_IMAGE_PROMPT,
)

Test the email agent with a natural language request:

In [189]:
image_url="https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120"
query = image_url

for step in web_image_subagent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_web_image (call_3dgzmCb1eivFZsKWx382SbuM)
 Call ID: call_3dgzmCb1eivFZsKWx382SbuM
  Args:
    image_url: https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120
================================= Tool Message =================================
Name: get_web_image

The image shows a silver BMW car driving on a road with a scenic backdrop. The car is captured from the front, showcasing its sleek design and distinctive kidney grille. The headlights are on, adding to the dynamic appearance. The license plate reads "M IA 1272E." The background features a blurred view of rocky terrain and a clear blue sky, suggesting motion and speed. The overall composition emphasizes the car's sporty and modern aesthetic.
================================== Ai Message ==================================

The image depicts a silver BMW car driving on a road with a scenic b

#### Create the supervisor agent

##### Wrap sub-agents as tools

Now wrap each sub-agent as a tool that the supervisor can invoke. This is the key architectural step that creates the layered system. The supervisor will see high-level tools like "web_article_event", not low-level tools like "get_web_article".<p>
The tool descriptions help the supervisor decide when to use each tool, so make them clear and specific. We return only the sub-agent’s final response, as the supervisor doesn’t need to see intermediate reasoning or tool calls.

In [23]:
@tool
def web_article_event(request: str) -> str:
    """
    Web article events
    Use this when the user gives an url for an article
    Help user to get accurate information from the given website url"
    """
    result = web_article_subagent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


@tool
def web_image_event(request: str) -> str:
    """
    Web image events
    Use this when the user gives an url for an image
    Help user to get accurate detail description of the image from the given website url"
    """
    result = web_image_subagent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content



##### Create the supervisor agent with high-level tools

Now create the supervisor that orchestrates the sub-agents. The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual API level.

In [24]:
SUPERVISOR_PROMPT = (
    """
    You are a helpful AI assistant.
    You can help users get detailed information when given a website URL.
    Choose exactly one tool based on whether the URL is an image or an article.
    When a tool is called, return ONLY the tool’s output as the final answer—do not add extra commentary.
    """
)

supervisor_agent = create_agent(
    model,
    tools=[web_article_event, web_image_event],
    system_prompt=SUPERVISOR_PROMPT,
)



#### Test the supervisor agent

Now test your complete system with complex requests that require coordination across multiple domains:

In [ ]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
image_url= "https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120" 

query =article_url

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()


================================== Ai Message ==================================
Tool Calls:
  web_article_event (call_AmGkfWgqOTYQD7lKKxA7Zq0o)
 Call ID: call_AmGkfWgqOTYQD7lKKxA7Zq0o
  Args:
    request: https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/
================================= Tool Message =================================
Name: web_article_event

Here is the content from the provided URL and its translation into Chinese:

**Original Content:**

Die Deutsche Umwelthilfe im Netz Pressemitteilung
• Mehr ultrafeine Feinstaub-Partikel und Stickoxide: DUH-Abgasmessung im realen Straßenbetrieb an Euro-5-Diesel-Pkw widerlegt Mythos von „besonders nachhaltigem“ Dieselkraftstoff HVO100
• DUH fordert Verkehrsminister Wissing auf, seine Behauptungen zu unterlassen, dass mit HVO100 „lokale Umweltbelastung in Städten und Kommunen“ reduziert werde, der Minister muss ihm vorl

In [192]:
article_url="https://www.duh.de/presse/pressemitteilungen/pressemitteilung/hvo100-noch-schmutziger-als-herkoemmlicher-diesel-abgasmessungen-der-deutschen-umwelthilfe-zerstoeren/"
image_url= "https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120" 

query =image_url

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()


================================== Ai Message ==================================
Tool Calls:
  web_image_event (call_Z38ILqufqxcMVROQXm8W7xGf)
 Call ID: call_Z38ILqufqxcMVROQXm8W7xGf
  Args:
    request: https://bmw.scene7.com/is/image/BMW/g26_bev_mp_positioning_image:16to7?fmt=webp&wid=2560&hei=1120
================================= Tool Message =================================
Name: web_image_event

The image depicts a silver BMW car driving on a road with a scenic backdrop. The car is captured from the front, highlighting its sleek design and distinctive kidney grille. The headlights are illuminated, and the license plate reads "M IA 1272E." The road is flanked by rocky terrain on the left, with a mountain visible in the background under a partly cloudy sky. The motion blur effect in the image suggests that the car is moving at a high speed.
================================== Ai Message ==================================

The image depicts a silver BMW car driving on a road with a 

##### Example 1: Simple single-domain request

In [18]:
query = "Schedule a team standup for tomorrow at 9am with Alice and Bob."

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_BToP7lMLpQW6tFlzcawpzzPj)
 Call ID: call_BToP7lMLpQW6tFlzcawpzzPj
  Args:
    request: Schedule a team standup for tomorrow at 9am with Alice and Bob.
================================= Tool Message =================================
Name: schedule_event

The team standup has been scheduled for tomorrow, October 31st, from 9:00 AM to 9:30 AM with Alice and Bob.
================================== Ai Message ==================================

The team standup has been successfully scheduled for tomorrow, October 31st, from 9:00 AM to 9:30 AM with Alice and Bob.


The supervisor identifies this as a calendar task, calls schedule_event, and the calendar agent handles date parsing and event creation.

##### Example 2: Complex multi-domain request

In [20]:
query = (
    '''
    Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, 
    and then send them an email reminder about reviewing the new mockups.
    '''
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_CbBD5ifQXYQMNqOTjEbfPOIc)
 Call ID: call_CbBD5ifQXYQMNqOTjEbfPOIc
  Args:
    request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_xd7CH6ZF7SO3FjAc0TDwCciA)
 Call ID: call_xd7CH6ZF7SO3FjAc0TDwCciA
  Args:
    request: Send an email reminder to the design team about reviewing the new mockups
================================= Tool Message =================================
Name: schedule_event

The meeting with the design team has been scheduled for next Tuesday, November 7th, from 2:00 PM to 3:00 PM.
================================= Tool Message =================================
Name: manage_email

I have sent the email reminder to the design team regarding the review of the new mockups. The subject of the email was "Reminder: Review New Mockups," and it was sent to design-team@example.com.
================================== 

The supervisor recognizes this requires both calendar and email actions, calls schedule_event for the meeting, then calls manage_email for the reminder. Each sub-agent completes its task, and the supervisor synthesizes both results into a coherent response.

In [23]:
response = model.invoke([
  HumanMessage("What is machine learning?")
])
print(response.usage_metadata)

{'input_tokens': 12, 'output_tokens': 304, 'total_tokens': 316, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [26]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

||Par|rots| have| colorful| feathers| for| several| reasons|,| primarily| related| to| survival| and| reproduction|:

|1|.| **|M|ating| and| Attraction|**|:| Bright| and| colorful| feathers| are| often| used| to| attract| mates|.| In| many| par|rot| species|,| vibrant| plum|age| is| a| sign| of| health| and| genetic| fitness|,| making| individuals| with| such| traits| more| attractive| to| potential| mates|.

|2|.| **|Species| and| Individual| Recognition|**|:| The| distinct| color| patterns| help| parro|ts| recognize| members| of| their| own| species|,| which| is| important| for| social| interactions| and| breeding|.| It| can| also| help| individuals| recognize| each| other| within| fl|ocks|.

|3|.| **|Cam|ouflage|**|:| Although| it| might| seem| counter|int|uitive|,| the| bright| colors| can| actually| serve| as| camouflage| in| their| natural| habitats|.| Many| parro|ts| live| in| tropical| environments| where| the| play| of| light| through| the| canopy| and| the| colorful| surround

In [32]:
respsonse = model.stream("Why do parrots have colorful feathers?")
print(respsonse)

<generator object BaseChatModel.stream at 0x7f9b2d628b80>
